## Figure 4b — Oscillation frequency and spatial wave number

This notebook reproduces Fig. 4b by comparing mean-field predictions with spiking-network simulations across different levels of external input.

- **Oscillation frequency:** Estimated from the power spectrum of the population firing rate.
- **Spatial wave number:** Estimated from plane fits to band-pass-filtered LFP phase maps. Wave events are identified using a fixed spatial coherence threshold (`FIXED_R_THRESHOLD`).
- **Comparison:** Simulation estimates are plotted against mean-field predictions, with variability across simulation trials.

The notebook generates `Fig4b_upper.svg` (wave number) and `Fig4b_lower.svg` (frequency), and saves the estimated wave statistics and diagnostic data in `.npz` files.

**Usage:** Set `DATA_DIR` to the simulation data directory and `OUTPUT_DIR` to the directory containing the mean-field predictions, then run the cells sequentially.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
import scipy.io as sio
from scipy import signal
from pathlib import Path



# ============================================================
# Load saved mean-field prediction
# ============================================================

OUTPUT_DIR = Path("MF_prediction_g3.3") #these are the results of the notebook "Fig4_a_b" in folder Theory/Fig4/
 
data = np.load(OUTPUT_DIR / "spatial_stability_summary.npz")
x_points = data["x_points"]
k_max_list = data["k_max_list"]
omega_at_kmax = data["omega_at_kmax"]

# ============================================================
# Load results from simulations
# ============================================================

DATA_DIR = Path("Isotropic_g3.3")
DRIVE_FACTORS = [0.77, 0.8275, 0.885, 0.9425, 1] 
N_ITERATIONS = 10
N_LFP_ITERATIONS = 10
TRIALS = [1,2,3,4,5,6]
 
# Change this if the rate monitor was sampled at a different interval.
SAMPLE_INTERVAL_MS = 0.1

# The spectral maximum is searched only in this range; change it if needed.
FREQUENCY_BAND_HZ = (5.0, 60.0)


FIXED_R_THRESHOLD = 0.4
FIXED_RESIDUAL_THRESHOLD_RAD=10 #not used for the selection any longer


LFP_BAND_HALF_WIDTH_HZ = 3.0


NameError: name 'yq' is not defined

This version uses a fixed spatial resultant cutoff, with no surrogate calibration. Spatial r measures gradient-direction alignment across valid electrodes at each frame. The event-level `direction_resultant` measures temporal direction concentration and remains a diagnostic only.


In [ ]:
# ============================================================
# LFP wave-number analysis: k = 2*pi / wavelength
# ============================================================
# Plane-fit wave number with separate temporal direction diagnostics.
# Detection uses spatial unit-gradient resultant r and plane-fit residual.

# LFP files: LFP_i<iteration>_J<one-based drive index>_trial<seed>.mat.
# Keep this separate from N_ITERATIONS, which is used by the rate files.

LFP_SAMPLE_INTERVAL_MS = 1.0
ELECTRODE_SPACING_MM = 0.141   

LFP_BAND_SEARCH_HZ = (5.0, 50.0)
LFP_BAND_LIMITS_HZ = (5.0, 30.0)


# Adjustable coherence cutoff, not a calibrated significance level.
# Use 0.6 to match the other fixed-r notebooks; 0.95 is a stricter example.
# Old Z > 90 means r > sqrt(90/N_valid), not one universal r cutoff.


if not np.isfinite(FIXED_R_THRESHOLD) or not 0 <= FIXED_R_THRESHOLD < 1:
    raise ValueError("FIXED_R_THRESHOLD must satisfy 0 <= r < 1")

LFP_FILTER_ORDER = 4
MIN_WAVE_DURATION_MS = 10.0
MIN_GRADIENT_MM_INV = 1e-3
WAVE_RANDOM_SEED = 0
WAVE_SUMMARY_OUTPUT = DATA_DIR / f"wave_plane_gradient_summary_r{FIXED_R_THRESHOLD:g}.npz"
K_ERRORBAR = "sd"  # "sd" (between-seed spread) or "ci95" (approximate t interval)

SMOOTH_WIDTH_MS = 5.0             # width passed to smooth_rate() in the simulation
EDGE_MODE = "trim"                # "trim" (drop + bridge) or "renormalise" (keep all samples)
TRIM_MS = 5.0                     # removed on each side of every chunk boundary ("trim" mode)
FREQUENCY_BAND_HZ = (5.0, 60.0)
WELCH_SEGMENT_S = 1.0             # 1 Hz resolution; 2.5 s record -> 4 averaged segments
ZERO_PAD_FACTOR = 8               # interpolates the spectrum (no new resolution)
FIT_HALF_WIDTH_HZ = 1.5           # parabola is fitted to log-power within +/- this
BACKGROUND_EXCLUDE_HZ = 3.0       # excluded around the peak when fitting 1/f background
MIN_PROMINENCE = 6.0              # peak/background power; noise-only 99th pct is ~5 (see not

def lfp_filename(iteration, drive_index, trial):
    return DATA_DIR / "LFP_i{}_J{}_trial{}.mat".format(
        iteration, drive_index + 1, trial
    )


def load_lfp_trial(drive_index, trial):
    """Load, concatenate, and channel-wise z-score i1...i5 for one trial."""
    chunks = []
    n_channels = None

    for iteration in range(1, N_LFP_ITERATIONS + 1):
        path = lfp_filename(iteration, drive_index, trial)
        if not path.exists():
            raise FileNotFoundError("Missing LFP file: {}".format(path))

        mat_data = sio.loadmat(path)
        if "data" not in mat_data:
            raise KeyError("{} has no 'data' variable.".format(path.name))
        chunk = np.asarray(mat_data["data"], dtype=float)
        if chunk.ndim != 2:
            raise ValueError(
                "{} must contain [time x channels] data, got {}."
                .format(path.name, chunk.shape)
            )
        if not np.all(np.isfinite(chunk)):
            raise ValueError("{} contains NaN or infinite values.".format(path.name))
        if n_channels is None:
            n_channels = chunk.shape[1]
        elif chunk.shape[1] != n_channels:
            raise ValueError("LFP channel count changes within J{}, trial {}.".format(
                drive_index + 1, trial
            ))
        chunks.append(chunk)

    lfp = np.concatenate(chunks, axis=0)
    channel_std = np.std(lfp, axis=0, keepdims=True)
    channel_std[channel_std == 0] = 1.0
    return (lfp - np.mean(lfp, axis=0, keepdims=True)) / channel_std


def grid_side(n_channels):
    n_side = int(round(np.sqrt(n_channels)))
    if n_side * n_side != n_channels:
        raise ValueError(
            "The MATLAB channel map requires a square array, not {} channels."
            .format(n_channels)
        )
    return n_side


def condition_band(lfp_trials):
    """Choose the common LFP band from the mean PSD across trials/channels."""
    fs_hz = 1000.0 / LFP_SAMPLE_INTERVAL_MS
    nperseg = min(1000, min(lfp.shape[0] for lfp in lfp_trials.values()))
    if nperseg < 4:
        raise ValueError("LFP recordings are too short for spectral estimation.")

    mean_psd = None
    frequencies = None
    for lfp in lfp_trials.values():
        frequencies, psd = signal.welch(
            lfp,
            fs=fs_hz,
            nperseg=nperseg,
            noverlap=nperseg // 2,
            nfft=max(2048, nperseg),
            axis=0,
        )
        channel_mean_psd = psd.mean(axis=1)
        mean_psd = channel_mean_psd if mean_psd is None else mean_psd + channel_mean_psd
    mean_psd /= len(lfp_trials)

    search = (
        (frequencies >= LFP_BAND_SEARCH_HZ[0])
        & (frequencies <= LFP_BAND_SEARCH_HZ[1])
    )
    if not np.any(search):
        raise ValueError("No PSD frequencies lie in LFP_BAND_SEARCH_HZ.")
    dominant_hz = frequencies[search][np.argmax(mean_psd[search])]
    band = (
        max(LFP_BAND_LIMITS_HZ[0], dominant_hz - LFP_BAND_HALF_WIDTH_HZ),
        min(LFP_BAND_LIMITS_HZ[1], dominant_hz + LFP_BAND_HALF_WIDTH_HZ),)


  #  band = (18.0, 21.0)     
    

    if not (0 < band[0] < band[1] < fs_hz / 2):
        raise ValueError("Invalid LFP band {} around {:.2f} Hz.".format(band, dominant_hz))
    return dominant_hz, band


def phase_maps_from_lfp(lfp, band_hz):
    """Filter, Hilbert-transform, map channels, then spatially unwrap phases."""
    fs_hz = 1000.0 / LFP_SAMPLE_INTERVAL_MS
    n_time, n_channels = lfp.shape
    n_side = grid_side(n_channels)
    sos = signal.butter(
        LFP_FILTER_ORDER, band_hz, btype="bandpass", fs=fs_hz, output="sos"
    )
    analytic = signal.hilbert(signal.sosfiltfilt(sos, lfp, axis=0), axis=0)
    phase_by_channel = np.angle(analytic)

    # Equivalent to MATLAB's ascending-down-columns channel_map.
    phases = phase_by_channel.reshape(n_time, n_side, n_side).transpose(0, 2, 1)
    phases = np.unwrap(np.unwrap(phases, axis=1), axis=2)
    phases -= phases[:, n_side // 2, n_side // 2][:, None, None]
    return phases.astype(np.float32, copy=False)


def plane_design(n_side):
    x_mm, y_mm = np.meshgrid(
        np.arange(n_side) * ELECTRODE_SPACING_MM,
        np.arange(n_side) * ELECTRODE_SPACING_MM,
    )
    design = np.column_stack((x_mm.ravel(), y_mm.ravel(), np.ones(n_side**2)))
    return design, np.linalg.pinv(design)


def phase_metrics(phases, design, pinv_design):
    """Spatial resultant r, residual, and plane-fit slopes in rad/mm."""
    grad_y, grad_x = np.gradient(
        phases,
        ELECTRODE_SPACING_MM,
        ELECTRODE_SPACING_MM,
        axis=(1, 2),
        edge_order=1,
    )
    grad_norm = np.hypot(grad_x, grad_y)
    valid = grad_norm > MIN_GRADIENT_MM_INV
    n_valid = valid.sum(axis=(1, 2)).astype(float)

    unit_x = np.divide(grad_x, grad_norm, out=np.zeros_like(grad_x), where=valid)
    unit_y = np.divide(grad_y, grad_norm, out=np.zeros_like(grad_y), where=valid)
    mean_x = np.divide(
        unit_x.sum(axis=(1, 2)),
        n_valid,
        out=np.zeros_like(n_valid),
        where=n_valid > 0,
    )
    mean_y = np.divide(
        unit_y.sum(axis=(1, 2)),
        n_valid,
        out=np.zeros_like(n_valid),
        where=n_valid > 0,
    )
    spatial_r = np.clip(np.hypot(mean_x, mean_y), 0.0, 1.0)

    phase_flat = phases.reshape(phases.shape[0], -1)
    coefficients = phase_flat @ pinv_design.T
    fitted = coefficients @ design.T
    residual = np.sqrt(np.mean((phase_flat - fitted) ** 2, axis=1))
    return spatial_r, residual, grad_x, grad_y, coefficients[:, 0], coefficients[:, 1]


def wave_wavelengths_and_wavenumbers(r, residual, plane_kx, plane_ky,
                                      r_threshold, residual_threshold):
    """One estimate per detected event: mean_t hypot(a(t), b(t)).

    Equal frame weighting within events; equal event weighting within seeds.
    Direction statistics are diagnostic only, never extra selection criteria.
    No minimum event k is imposed. Exactly zero k has infinite wavelength.
    """
 #   detected = (r > r_threshold) & (residual < residual_threshold)

    detected = (r > r_threshold)  
    
    minimum_samples = max(1, int(round(MIN_WAVE_DURATION_MS / LFP_SAMPLE_INTERVAL_MS)))
    padded = np.r_[False, detected, False]
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    names = ["start_frame", "stop_frame_exclusive", "duration_ms", "k_mm_inv",
             "wavelength_mm", "vector_mean_k_mm_inv", "cancellation_ratio",
             "direction_resultant", "mean_gradient_angle_rad",
             "direction_valid_fraction", "mean_residual_rad"]
    records = {name: [] for name in names}
    for start, stop in zip(changes[0::2], changes[1::2]):
        if stop - start < minimum_samples:
            continue
        a, b = plane_kx[start:stop], plane_ky[start:stop]
        if not (np.all(np.isfinite(a)) and np.all(np.isfinite(b))):
            raise ValueError("Non-finite plane-fit slopes in a detected event")
        frame_k = np.hypot(a, b)
        event_k = float(frame_k.mean())
        vector_k = float(np.hypot(a.mean(), b.mean()))
        # The existing numerical floor is used ONLY to define directions here.
        valid_direction = frame_k > MIN_GRADIENT_MM_INV
        resultant = mean_angle = np.nan
        if np.any(valid_direction):
            ux = np.mean(a[valid_direction] / frame_k[valid_direction])
            uy = np.mean(b[valid_direction] / frame_k[valid_direction])
            resultant = float(np.clip(np.hypot(ux, uy), 0, 1))
            if resultant > np.finfo(float).eps:
                mean_angle = float(np.arctan2(uy, ux))
        values = [start, stop, (stop-start)*LFP_SAMPLE_INTERVAL_MS, event_k,
                  2*np.pi/event_k if event_k > 0 else np.inf, vector_k,
                  float(np.clip(vector_k/event_k, 0, 1)) if event_k > 0 else np.nan,
                  resultant, mean_angle, valid_direction.mean(), residual[start:stop].mean()]
        for name, value in zip(names, values):
            records[name].append(value)
    return {name: np.asarray(values, dtype=int if name in
            ("start_frame", "stop_frame_exclusive") else float)
            for name, values in records.items()}


def analyse_drive(drive_index, rng):
    """Fixed detection thresholds; no shuffled surrogates.

    rng is retained for compatibility with the existing calling code.
    """
    lfp_trials = {
        trial: load_lfp_trial(drive_index, trial)
        for trial in TRIALS
    }
    dominant_hz, band_hz = condition_band(lfp_trials)

    r_threshold = FIXED_R_THRESHOLD
    residual_threshold = FIXED_RESIDUAL_THRESHOLD_RAD

    analyses = {}
    results = {}

    for trial, lfp in lfp_trials.items():
        phases = phase_maps_from_lfp(lfp, band_hz)
        design, pinv_design = plane_design(phases.shape[1])

        r, residual, _, _, a, b = phase_metrics(
            phases, design, pinv_design
        )

        analyses[trial] = dict(
            r=r, residual=residual, kx=a, ky=b
        )

        results[trial] = wave_wavelengths_and_wavenumbers(
            r, residual, a, b,
            r_threshold, residual_threshold
        )

    return (
        dominant_hz,
        band_hz,
        r_threshold,
        residual_threshold,
        results,
        analyses,
    )
def trial_mean_and_sd(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.nan, np.nan
    if finite.size == 1:
        return finite[0], np.nan
    return finite.mean(), finite.std(ddof=1)


def seed_uncertainty(values):
    """Descriptive SD and approximate Student-t CI over finite seed means.

    Events/frames are not independent replicates. This interval is conditional
    on the fitted detector/band and does not correct positive magnitude bias.
    It is NOT a test against a synchronous/noise null.
    """
    from scipy.stats import t as student_t
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    mean, sd = trial_mean_and_sd(values)
    count = values.size
    sem = sd / np.sqrt(count) if count >= 2 else np.nan
    half = student_t.ppf(0.975, count-1)*sem if count >= 2 else np.nan
    return mean, sd, sem, mean-half, mean+half, count


n_drives, n_trials = len(DRIVE_FACTORS), len(TRIALS)
simulated_k_trial_mean_mm_inv = np.full((n_drives, n_trials), np.nan)
simulated_wavelength_trial_mean_mm = np.full((n_drives, n_trials), np.nan)
simulated_direction_trial_mean_resultant = np.full((n_drives, n_trials), np.nan)
detected_wave_count = np.zeros((n_drives, n_trials), dtype=int)
lfp_dominant_frequency_hz = np.full(n_drives, np.nan)
lfp_filter_band_hz = np.full((n_drives, 2), np.nan)
r_thresholds = np.full(n_drives, np.nan)
residual_thresholds = np.full(n_drives, np.nan)
# Small frame-level diagnostic arrays are retained, not the full phase maps.
wave_events, wave_frame_diagnostics = {}, {}
event_parts = []
rng = np.random.default_rng(WAVE_RANDOM_SEED)
for drive_index, drive_factor in enumerate(DRIVE_FACTORS):
    dominant_hz, band_hz, r_threshold, residual_threshold, results, frames = analyse_drive(drive_index, rng)
    lfp_dominant_frequency_hz[drive_index] = dominant_hz
    lfp_filter_band_hz[drive_index] = band_hz
    r_thresholds[drive_index] = r_threshold
    residual_thresholds[drive_index] = residual_threshold
    for trial_position, trial in enumerate(TRIALS):
        events = results[trial]
        wave_events[(drive_index, trial)] = events
        wave_frame_diagnostics[(drive_index, trial)] = frames[trial]
        count = events["k_mm_inv"].size
        detected_wave_count[drive_index, trial_position] = count
        if count:
            simulated_k_trial_mean_mm_inv[drive_index, trial_position] = events["k_mm_inv"].mean()
            simulated_wavelength_trial_mean_mm[drive_index, trial_position] = events["wavelength_mm"].mean()
            finite_r = events["direction_resultant"][np.isfinite(events["direction_resultant"])]
            if finite_r.size:
                simulated_direction_trial_mean_resultant[drive_index, trial_position] = finite_r.mean()
        event_parts.append(dict(events, drive_index=np.full(count, drive_index, dtype=int),
                                trial_number=np.full(count, trial, dtype=int)))

seed_stats = np.array([seed_uncertainty(row) for row in simulated_k_trial_mean_mm_inv])
(simulated_k_mean_mm_inv, simulated_k_std_mm_inv, simulated_k_sem_mm_inv,
 simulated_k_ci95_low_mm_inv, simulated_k_ci95_high_mm_inv, valid_seed_count) = seed_stats.T
valid_seed_count = valid_seed_count.astype(int)
# Secondary quantity: reciprocal of mean k, NOT mean of event wavelengths.
characteristic_wavelength_mm = np.divide(2*np.pi, simulated_k_mean_mm_inv,
    out=np.full(n_drives, np.inf), where=simulated_k_mean_mm_inv > 0)
characteristic_wavelength_mm[np.isnan(simulated_k_mean_mm_inv)] = np.nan

for d, factor in enumerate(DRIVE_FACTORS):
    print(f"J{d+1} (factor={factor}): mean plane k={simulated_k_mean_mm_inv[d]:.4f}; "
          f"seed SD={simulated_k_std_mm_inv[d]:.4f}; "
          f"approx. 95% t CI=[{simulated_k_ci95_low_mm_inv[d]:.4f}, "
          f"{simulated_k_ci95_high_mm_inv[d]:.4f}] mm^-1; "
          f"valid seeds={valid_seed_count[d]}/{n_trials}; events={detected_wave_count[d].tolist()}")
    print(f"  seed means: {simulated_k_trial_mean_mm_inv[d]}; "
          f"band={lfp_filter_band_hz[d]}; r threshold={r_thresholds[d]:.4f}; "
          f"residual threshold={residual_thresholds[d]:.4f} rad")
    if valid_seed_count[d] < n_trials:
        print("  Seeds without detected events are missing, not zero k; summaries are conditional on detection.")
    if valid_seed_count[d] < 3:
        print("  Fewer than 3 contributing seeds: uncertainty is poorly determined (CI undefined for n<2).")
print("CI assumes approximately normal independent seed means, conditional on the pooled detector/band. "
      "It does not quantify gradient measurement bias or establish a nonzero physical wave number.")

saved_events = {key: np.concatenate([part[key] for part in event_parts]) for key in event_parts[0]}
frame_save = {}
for (d, trial), frames in wave_frame_diagnostics.items():
    for key, values in frames.items():
        frame_save[f"frames_J{d+1}_trial{trial}_{key}"] = values
np.savez_compressed(
    WAVE_SUMMARY_OUTPUT,
    threshold_method="fixed_r",
    min_wave_duration_ms=MIN_WAVE_DURATION_MS,
    min_gradient_mm_inv=MIN_GRADIENT_MM_INV,
    estimator="event mean of framewise plane-fit gradient magnitudes; equal events within seeds, equal seeds",
    drive_factors=np.asarray(DRIVE_FACTORS), trial_numbers=np.asarray(TRIALS),
    mean_wavenumber_mm_inv=simulated_k_mean_mm_inv,
    std_wavenumber_mm_inv=simulated_k_std_mm_inv,
    sem_wavenumber_mm_inv=simulated_k_sem_mm_inv,
    ci95_low_wavenumber_mm_inv=simulated_k_ci95_low_mm_inv,
    ci95_high_wavenumber_mm_inv=simulated_k_ci95_high_mm_inv,
    valid_seed_count=valid_seed_count,
    trial_mean_wavenumber_mm_inv=simulated_k_trial_mean_mm_inv,
    characteristic_wavelength_mm=characteristic_wavelength_mm,
    trial_mean_event_wavelength_mm=simulated_wavelength_trial_mean_mm,
    trial_mean_direction_resultant=simulated_direction_trial_mean_resultant,
    detected_wave_count=detected_wave_count,
    lfp_dominant_frequency_hz=lfp_dominant_frequency_hz,
    lfp_filter_band_hz=lfp_filter_band_hz,
    r_thresholds=r_thresholds, residual_thresholds=residual_thresholds,
    lfp_sample_interval_ms=LFP_SAMPLE_INTERVAL_MS,
    electrode_spacing_mm=ELECTRODE_SPACING_MM,
    **{"event_"+key: value for key, value in saved_events.items()}, **frame_save,
)
print(f"Saved plane-gradient estimates and diagnostics to {WAVE_SUMMARY_OUTPUT}")


In [ ]:
def rate_filename(population, iteration, drive_index, trial):
    return DATA_DIR / f"RATE_{population}_i{iteration}_J{drive_index}_trial{trial}.mat"


def load_rate(path):
    data = sio.loadmat(path)
    if "data" not in data:
        raise KeyError(f"{path.name} has no 'data' variable.")

    rate = np.asarray(data["data"], dtype=float).squeeze()
    if rate.ndim != 1:
        raise ValueError(f"Expected a one-dimensional rate in {path.name}; got shape {rate.shape}.")
    return rate


def load_population_rate(drive_index, trial, rate_filename, load_rate, data_dir=None):
    """Chunk-aware version of the notebook's load_population_rate().

    `rate_filename` and `load_rate` are the notebook's existing helpers.
    Returns the 0.8*E + 0.2*I rate with chunk edges removed/bridged.
    """
    chunks = []
    for iteration in range(1, N_ITERATIONS + 1):
        e = load_rate(rate_filename("E", iteration, drive_index + 1, trial))
        i = load_rate(rate_filename("I", iteration, drive_index + 1, trial))
        if e.size != i.size:
            raise ValueError(f"E/I length mismatch (J{drive_index + 1}, trial {trial}, i{iteration})")
        chunks.append(0.8 * e + 0.2 * i)
    return clean_chunk_edges(chunks, SAMPLE_INTERVAL_MS)


def clean_chunk_edges(chunks, dt_ms, trim_ms=TRIM_MS, mode=EDGE_MODE,
                     smooth_width_ms=SMOOTH_WIDTH_MS):
    """Concatenate chunks and remove the smooth_rate() edge artifact.

    mode="trim":        drop +/- trim_ms around every interior chunk boundary and
                        bridge the gap by linear interpolation (keeps the time
                        base); also drop trim_ms at the start/end of the record.
    mode="renormalise": divide each edge sample by the fraction of the boxcar
                        window that lay inside the chunk (undoes the zero
                        padding, keeps every sample).
    """
    if mode == "renormalise":
        half = int(round(smooth_width_ms / (2 * dt_ms)))     # 25 samples for 5 ms @ 0.1 ms
        window = 2 * half + 1                                # 51 samples (Brian2's rule)
        coverage = (np.arange(half) + half + 1) / window     # 26/51 ... 50/51
        fixed = []
        for c in chunks:
            c = np.asarray(c, dtype=float).copy()
            c[:half] /= coverage
            c[-half:] /= coverage[::-1]
            fixed.append(c)
        return np.concatenate(fixed)

    n_trim = int(round(trim_ms / dt_ms))
    rate = np.concatenate(chunks).astype(float)
    n = rate.size
    boundaries = np.cumsum([c.size for c in chunks])[:-1]

    bad = np.zeros(n, dtype=bool)
    for b in boundaries:
        bad[max(0, b - n_trim): b + n_trim] = True
    good = np.flatnonzero(~bad)
    rate = np.interp(np.arange(n), good, rate[good])         # linear bridge over the gap
    return rate[n_trim: n - n_trim]


In [ ]:
population_rates = {}
for drive_index in range(len(DRIVE_FACTORS)):
    for trial in TRIALS:
        population_rates[(drive_index, trial)] = load_population_rate(
            drive_index, trial, rate_filename, load_rate)

In [ ]:
def dominant_frequency(rate_hz, dt_ms=SAMPLE_INTERVAL_MS, band_hz=FREQUENCY_BAND_HZ,
                       segment_s=WELCH_SEGMENT_S, min_prominence=MIN_PROMINENCE,
                       return_details=False):
    """Peak frequency (Hz) of the rate spectrum inside band_hz.

    Returns NaN when the peak is not `min_prominence` times above a power-law
    background fitted to the rest of the band.
    """
    fs = 1000.0 / dt_ms
    low, high = band_hz
    if not (0 < low < high < fs / 2):
        raise ValueError("band_hz must lie between 0 and Nyquist.")

    x = np.asarray(rate_hz, dtype=float)
    nperseg = min(int(round(segment_s * fs)), x.size)
    nfft = int(2 ** np.ceil(np.log2(nperseg * ZERO_PAD_FACTOR)))
    f, p = signal.welch(x, fs=fs, window="hann", nperseg=nperseg,
                        noverlap=nperseg // 2, nfft=nfft,
                        detrend="constant", scaling="density")

    in_band = (f >= low) & (f <= high)
    fb, pb = f[in_band], p[in_band]
    i_max = np.argmax(pb)
    f_coarse = fb[i_max]

    # parabolic fit to log-power around the maximum -> sub-bin peak location
    win = np.abs(fb - f_coarse) <= FIT_HALF_WIDTH_HZ
    f_peak = f_coarse
    if win.sum() >= 5:
        a, b, _ = np.polyfit(fb[win] - f_coarse, np.log(pb[win]), 2)
        if a < 0:
            vertex = -b / (2 * a)
            if abs(vertex) <= FIT_HALF_WIDTH_HZ:
                f_peak = f_coarse + vertex

    # prominence relative to a log-log linear background fitted away from the peak
    bg_mask = np.abs(fb - f_coarse) > BACKGROUND_EXCLUDE_HZ
    if bg_mask.sum() >= 10:
        slope, icpt = np.polyfit(np.log(fb[bg_mask]), np.log(pb[bg_mask]), 1)
        background = np.exp(icpt + slope * np.log(f_coarse))
        prominence = pb[i_max] / background
    else:
        prominence = np.nan

    freq = f_peak if (np.isnan(prominence) or prominence >= min_prominence) else np.nan
    if return_details:
        return freq, prominence, (f, p)
    return freq

In [ ]:
frequencies_by_drive = {}
for drive_index, drive_factor in enumerate(DRIVE_FACTORS):
    out = [dominant_frequency(population_rates[(drive_index, t)], return_details=True)
           for t in TRIALS]
    freqs = np.array([o[0] for o in out])
    prom  = np.array([o[1] for o in out])
    frequencies_by_drive[drive_index] = freqs
    print(f"J{drive_index + 1} ({drive_factor}): "
          f"{np.nanmean(freqs):.2f} +/- {np.nanstd(freqs, ddof=1):.2f} Hz, "
          f"valid trials {np.isfinite(freqs).sum()}/{len(TRIALS)}, "
          f"prominence {np.round(prom, 1)}")


In [ ]:
frequencies_by_drive

In [ ]:
# Dominant-frequency summary from the preceding notebook cells
drive_values = np.asarray(DRIVE_FACTORS, dtype=float)
frequency_mean = np.array([
    np.nanmean(frequencies_by_drive[j])
    for j in range(len(DRIVE_FACTORS))
])
frequency_std = np.array([
    np.nanstd(frequencies_by_drive[j], ddof=1)
    for j in range(len(DRIVE_FACTORS))
])

# One-panel frequency figure
fig, ax = plt.subplots(figsize=(7, 5))

 
ax.plot(
    x_points,
    np.abs(omega_at_kmax),
    ".k",
    markersize=6,  # <-- Added missing comma here
    # label=r"$\omega(k_{max})$ from mean field",  # <-- Added raw string (r) and dollar signs ($) for LaTeX rendering
)
 

 
ax.errorbar(
    drive_values,
    frequency_mean,
    yerr=frequency_std,
    fmt="o",
    color="tab:orange",
    capsize=4,
    markersize=10,
    # label="Simulations rates",
)
 
ax.set_xlim(0.7, 1.2)
ax.set_xlabel(r"$\nu_{\mathrm{ext}}/\nu_{\theta}$", fontsize=16)
ax.set_ylabel("Frequency (Hz)", fontsize=16)
ax.tick_params(labelsize=14)
 
fig.tight_layout()
fig.savefig("Fig4b_lower.svg", format="svg", bbox_inches="tight")
plt.show()
 

In [ ]:
frequency_mean

In [ ]:
# ============================================================
# Plot mean-field k_max and directly estimated plane-gradient magnitudes
# ============================================================

fig, ax = plt.subplots(figsize=(7, 5))


ax.plot(
    x_points,
    k_max_list,
    "k.",
    linewidth=2,)


    
if K_ERRORBAR not in ("sd", "ci95"):
    raise ValueError("K_ERRORBAR must be 'sd' or 'ci95'")
error = (simulated_k_std_mm_inv if K_ERRORBAR == "sd" else
         (simulated_k_ci95_high_mm_inv-simulated_k_ci95_low_mm_inv)/2)
error_label = "seed SD" if K_ERRORBAR == "sd" else "approx. 95% t CI"
for d, factor in enumerate(DRIVE_FACTORS):
    ax.scatter(np.full(n_trials, factor), simulated_k_trial_mean_mm_inv[d],
               color="tab:orange", alpha=0.5, marker="x",
               # label="Individual seed means" if d == 0 else None
              )
ax.errorbar(
    np.asarray(DRIVE_FACTORS, dtype=float),
    simulated_k_mean_mm_inv,
    yerr=error,
    fmt="o",
    color="tab:orange",
    capsize=4,
    markersize=7,
    # label=f"Plane-gradient mean ± {error_label}",
)

ax.set_xlabel(r"$\nu_{\mathrm{ext}}/\nu_{\theta}$", fontsize=16)
ax.set_ylabel(r"$k\;(\mathrm{mm}^{-1})$", fontsize=16)
ax.tick_params(labelsize=14)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=12)

fig.tight_layout()
fig.savefig("Fig4b_upper.svg", format="svg", bbox_inches="tight")
plt.show()
 


In [ ]:
# ============================================================
# Save all data needed to reproduce Fig. 4b upper panel
# ============================================================

import numpy as np

np.savez_compressed(
    f"Fig4b_upper_data_th{FIXED_R_THRESHOLD}.npz",

    # Mean-field prediction
    x_points=np.asarray(x_points),
    k_max_list=np.asarray(k_max_list),

 

    # Simulation results
    DRIVE_FACTORS=np.asarray(DRIVE_FACTORS),
    simulated_k_trial_mean_mm_inv=np.asarray(simulated_k_trial_mean_mm_inv),
    simulated_k_mean_mm_inv=np.asarray(simulated_k_mean_mm_inv),
    simulated_k_std_mm_inv=np.asarray(simulated_k_std_mm_inv),
    simulated_k_ci95_low_mm_inv=np.asarray(simulated_k_ci95_low_mm_inv),
    simulated_k_ci95_high_mm_inv=np.asarray(simulated_k_ci95_high_mm_inv),

    # Plot configuration
    K_ERRORBAR=K_ERRORBAR,
)

print("Saved: Fig4b_upper_data.npz")

In [ ]:
print("omega detected from rate:", list(zip(drive_values, frequency_mean)))
print("omega detected from LFP:", list(zip(drive_values, lfp_dominant_frequency_hz)))
print("k detected:",simulated_k_mean_mm_inv)

## Inspect direction stability and cancellation
These are descriptive checks, not extra event filters. Angles wrap at ±180 degrees. Event times are relative to the start of the saved recording, after warmup.


In [ ]:
# Event-level diagnostics: no new detection criteria.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for d, factor in enumerate(DRIVE_FACTORS):
    sel = saved_events["drive_index"] == d
    k = saved_events["k_mm_inv"][sel]
    axes[0].scatter(k, saved_events["vector_mean_k_mm_inv"][sel], s=18,
                    alpha=0.6, label=f"{factor:g}")
    axes[1].scatter(k, saved_events["direction_resultant"][sel], s=18, alpha=0.6)
finite_k = saved_events["k_mm_inv"][np.isfinite(saved_events["k_mm_inv"])]
limit = finite_k.max() if finite_k.size and finite_k.max() > 0 else 1.0
axes[0].plot([0, limit], [0, limit], "k--", lw=1)
axes[0].set(xlabel="Mean framewise plane |k| (mm⁻¹)",
            ylabel="Magnitude of event-mean plane vector (mm⁻¹)",
            title="Below diagonal: directional cancellation")
axes[0].legend(title="Input factor")
axes[1].set(xlabel="Mean framewise plane |k| (mm⁻¹)",
            ylabel="Temporal direction concentration R", ylim=(-0.03, 1.03),
            title="Direction stability is reported separately")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


def plot_wave_event(drive_index, trial, event_index=0):
    """Zero-based drive/event indices; trial is the actual seed number."""
    events = wave_events[(drive_index, trial)]
    frames = wave_frame_diagnostics[(drive_index, trial)]
    count = events["k_mm_inv"].size
    if not 0 <= event_index < count:
        raise ValueError(f"Requested event {event_index}; this seed has {count} events")
    start = events["start_frame"][event_index]
    stop = events["stop_frame_exclusive"][event_index]
    sl = slice(start, stop)
    time_ms = np.arange(start, stop)*LFP_SAMPLE_INTERVAL_MS
    a, b = frames["kx"][sl], frames["ky"][sl]
    k = np.hypot(a, b)
    angle = np.arctan2(b, a)
    angle[k <= MIN_GRADIENT_MM_INV] = np.nan
    fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
    axes[0].plot(time_ms, k, label="Framewise plane |k|")
    axes[0].axhline(events["k_mm_inv"][event_index], color="k", ls="--", label="Event mean |k|")
    axes[0].axhline(events["vector_mean_k_mm_inv"][event_index], color="tab:red", ls=":", label="|Event mean vector|")
    axes[0].set_ylabel("k (mm⁻¹)")
    axes[0].legend()
    axes[1].plot(time_ms, np.degrees(angle), ".")
    axes[1].set(ylabel="Gradient angle (degrees)", ylim=(-185, 185))
    axes[2].plot(time_ms, frames["residual"][sl], label="Plane-fit RMS residual")
    axes[2].axhline(residual_thresholds[drive_index], color="k", ls="--", label="Detection threshold")
    axes[2].set(xlabel="Time after warmup (ms)", ylabel="Residual (rad)")
    axes[2].legend()
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.suptitle(f"Input {DRIVE_FACTORS[drive_index]}, seed {trial}, event {event_index}: "
                 f"R={events['direction_resultant'][event_index]:.3f}, "
                 f"C={events['cancellation_ratio'][event_index]:.3f}")
    fig.tight_layout()
    plt.show()

# Display the first accepted event; change arguments to inspect others.
first_event = next(((d, trial) for (d, trial), e in wave_events.items() if e["k_mm_inv"].size), None)
if first_event is not None:
    plot_wave_event(*first_event, event_index=0)
else:
    print("No accepted events to display.")
